<a href="https://colab.research.google.com/github/ducnguyen170505-del/Sankey-Analysis/blob/main/BTVN_bu%E1%BB%95i_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import library

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.graph_objects as go

# 2. Import data and Preprocessing

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
order_item = pd.read_csv('/content/drive/MyDrive/Data a Tú/Buổi 9/BTVN/order_items.csv')
order = pd.read_csv('/content/drive/MyDrive/Data a Tú/Buổi 9/BTVN/orders.csv')
product = pd.read_csv('/content/drive/MyDrive/Data a Tú/Buổi 9/BTVN/products.csv')
pageview = pd.read_csv('/content/drive/MyDrive/Data a Tú/Buổi 9/BTVN/website_pageviews.csv')
session = pd.read_csv('/content/drive/MyDrive/Data a Tú/Buổi 9/BTVN/website_sessions.csv')

# 3. Sankey Analysis

## 3.1 Step 1
Bước 1: Xác định Landing Page
Trong bảng website_pageviews, một session có thể xem nhiều trang. Bạn cần xác định trang đầu tiên (dựa trên thời gian created_at) của mỗi website_session_id. Đây chính là Landing Page.

In [79]:
landing_pages = pageview.sort_values(['website_session_id', 'created_at']).groupby('website_session_id').first()['pageview_url'].reset_index()
landing_pages.columns = ['website_session_id', 'landing_page']
landing_pages.head()

,website_session_id,landing_page
0,1,/home
1,2,/home
2,3,/home
3,4,/home
4,5,/home


## 3.2 Step 2
Bước 2: Hợp nhất dữ liệu (Multi-table Join)
Kết nối các bảng lại với nhau để tạo thành một bảng "Master Journey" chứa:
utm_source (từ bảng website_sessions)
landing_page (vừa tìm được ở Bước 1)
product_name (từ bảng orders và products)


In [80]:
journey = session[['website_session_id', 'utm_source']].merge(landing_pages, on='website_session_id')
journey.head()

,website_session_id,utm_source,landing_page
0,1,gsearch,/home
1,2,gsearch,/home
2,3,gsearch,/home
3,4,gsearch,/home
4,5,gsearch,/home


In [83]:
journey = journey.merge(order[['website_session_id', 'primary_product_id']], on='website_session_id', how='left')
journey = journey.merge(product[['product_id', 'product_name']], left_on='primary_product_id', right_on='product_id', how='left')
journey.head()

KeyError: 'primary_product_id'

## 3.3 Step 3
Bước 3: Xử lý dữ liệu khách thoát (Drop-off)
Vì không phải khách hàng nào vào web cũng mua hàng, bạn cần thực hiện Left Join với bảng đơn hàng. Những session nào không có order_id sẽ được gán nhãn sản phẩm là "None (Drop-off)".
--> Đã làm


In [85]:
journey['product_name'] = journey['product_name'].fillna('None (Drop-off)')
journey = journey.drop(columns=['primary_product_id_x', 'primary_product_id_y'])
journey.head()

,website_session_id,utm_source,landing_page,product_id,product_name
0,1,gsearch,/home,NaN,None (Drop-off)
1,2,gsearch,/home,NaN,None (Drop-off)
2,3,gsearch,/home,NaN,None (Drop-off)
3,4,gsearch,/home,NaN,None (Drop-off)
4,5,gsearch,/home,NaN,None (Drop-off)


## 4. Step 4
 Tạo cấu trúc dữ liệu cho Sankey đa tầng
Biểu đồ Sankey cần dữ liệu dạng cặp Source - Target. Bạn cần chuẩn bị 2 tập dữ liệu và nối chồng (concat) chúng lại:
Tầng 1: utm_source (Source) $\rightarrow$ landing_page (Target)
Tầng 2: landing_page (Source) $\rightarrow$ product_name (Target)


In [86]:
l1 = journey.groupby(['utm_source', 'landing_page']).size().reset_index(name='value')
l1.columns = ['source', 'target', 'value']
l1.head()

,source,target,value
0,bsearch,/home,7914
1,bsearch,/lander-1,9459
2,bsearch,/lander-2,24076
3,bsearch,/lander-3,6178
4,bsearch,/lander-4,1903


In [87]:
l2 = journey.groupby(['landing_page', 'product_name']).size().reset_index(name='value')
l2.columns = ['source', 'target', 'value']
l2.head()

,source,target,value
0,/home,None (Drop-off),127865
1,/home,The Birthday Sugar Panda,1003
2,/home,The Forever Love Bear,1458
3,/home,The Hudson River Mini bear,182
4,/home,The Original Mr. Fuzzy,7068


In [89]:
sankey_hw = pd.concat([l1, l2], axis=0)
sankey_hw

,source,target,value
0,bsearch,/home,7914
1,bsearch,/lander-1,9459
2,bsearch,/lander-2,24076
3,bsearch,/lander-3,6178
4,bsearch,/lander-4,1903
5,bsearch,/lander-5,13293
6,gsearch,/home,46334
7,gsearch,/lander-1,38115
8,gsearch,/lander-2,100982
9,gsearch,/lander-3,68249


In [90]:
all_nodes = list(pd.concat([sankey_hw['source'], sankey_hw['target']]).unique())
all_nodes

['bsearch',
 'gsearch',
 'socialbook',
 '/home',
 '/lander-1',
 '/lander-2',
 '/lander-3',
 '/lander-4',
 '/lander-5',
 'None (Drop-off)',
 'The Birthday Sugar Panda',
 'The Forever Love Bear',
 'The Hudson River Mini bear',
 'The Original Mr. Fuzzy']

In [91]:
node_map = {node: i for i, node in enumerate(all_nodes)}
node_map

{'bsearch': 0,
 'gsearch': 1,
 'socialbook': 2,
 '/home': 3,
 '/lander-1': 4,
 '/lander-2': 5,
 '/lander-3': 6,
 '/lander-4': 7,
 '/lander-5': 8,
 'None (Drop-off)': 9,
 'The Birthday Sugar Panda': 10,
 'The Forever Love Bear': 11,
 'The Hudson River Mini bear': 12,
 'The Original Mr. Fuzzy': 13}

In [69]:
sankey_data = pd.concat([layer1, layer2])
sankey_data

,source,landingpage,value,target
0,bsearch,2537.0,1,NaN
1,bsearch,3146.0,1,NaN
2,bsearch,4014.0,1,NaN
3,bsearch,4813.0,1,NaN
4,bsearch,6020.0,1,NaN
...,...,...,...,...
472866,1188111,NaN,1,None(Drop-off)
472867,1188114,NaN,1,None(Drop-off)
472868,1188116,NaN,1,None(Drop-off)
472869,1188118,NaN,1,None(Drop-off)


## 3.5 Step 5
Bước 5: Trực quan hóa (Visualization)
Sử dụng thư viện Plotly để vẽ biểu đồ Sankey. Biểu đồ phải thể hiện được sự liền mạch từ Nguồn $\rightarrow$ Trang đích $\rightarrow$ Sản phẩm.


In [92]:

fig = go.Figure(data=[go.Sankey(
    node = dict(pad=15, thickness=20, label=all_nodes, color='darkblue'),
    link = dict(
        source = sankey_hw['source'].map(node_map),
        target = sankey_hw['target'].map(node_map),
        value = sankey_hw['value']
    ))])

fig.update_layout(title_text="Web Journey Sankey: Source -> Landing Page -> Purchased Product", font_size=10)
fig.show()

### 1. Phân tích Hiệu quả Nguồn Traffic (Traffic Source Performance)

- Về mặt volume: google search đang chiếm tỷ lệ áp đảo và là kênh được mọi người sử dụng để truy cập thông tin. Tiếp theo là bsearch tuy traffic truy cập ít hơn nhiều so với google search nhưng đây vẫn là 1 kênh tiềm năng có thể khai thác thêm. Cuối cùng social book thường không đem lại hiệu quả cao -> cân nhắc loại bỏ hoặc chỉ tập trung vào quảng cáo 1 số loại sản phẩm có khả năng thu hút KH qua kênh này

### 2. Tối ưu hóa Trang đích (Landing Page Optimization)

- Home và lander 2 là 2 trang landingpage được bấm vào nhiều nhất và có volume ngang nhau. Nội dung và hình thức cũng có thể coi là thiết kế phù hợp vì là 2 kênh dẫn tới sản phẩm nhiều nhất

- Lander 3 và 5 là 2 kênh tiếp theo có số lượng truy cập cao nhát. Nhưng tỷ lệ KH chuyển sang pageview khác để mua sản phẩm thì lại không có. Điều này cho thấy 2 kênh này có tiềm năng phát triển thêm và cần cải thiện nội dung để thu hút người mua dễ dàng hơn
- Lander 4 và 1 là 2 kênh yếu nhất. Cân nhắc loại bỏ để tối ưu chi phí

### 3. Phân tích Product Holdings & Hành vi mua hàng

* Nhìn vào mặt hàng thì có thể thấy rằng sản phẩm này không phải sản phẩm tiêu dùng thường xuyên --> giải thích cho việc tại sao tỷ lệ drop off ở cả 5 kênh lại tương đối cao
* Original đang là sản phẩm chủ lực gánh về mặt doanh thu, vượt xa những sản phẩm còn lại. Đòi hỏi cần lưu kho mặt hàng này nhiều để đảm bảo không bị cháy hàng
* Một chiến lược có thể cân nhắc đến là tổ chức có thể tổ chức các sự kiện đặc biệt theo tháng như sinh nhật, sales để thu hút người mua hoặc nếu mua sản phẩm orginal thì có thể nhận giảm giá từ các sản phẩm khác để tăng volume bán